In [24]:
from datasets import load_dataset

ds = load_dataset("Teklia/IAM-line")
print(ds)

# Look at one sample
sample = ds['train'][0]
print(f"Text:         {sample['text']}")
print(f"Image size:   {sample['image'].size}")
print(f"Image mode:   {sample['image'].mode}")

DatasetDict({
    train: Dataset({
        features: ['image', 'text'],
        num_rows: 6482
    })
    validation: Dataset({
        features: ['image', 'text'],
        num_rows: 976
    })
    test: Dataset({
        features: ['image', 'text'],
        num_rows: 2915
    })
})
Text:         put down a resolution on the subject
Image size:   (2467, 128)
Image mode:   L


In [25]:
sample = ds['train'][0]
print(sample.keys())


dict_keys(['image', 'text'])


In [26]:
sample = ds['train'][0]
print(type(sample['image']))
print(sample['image'].info)
print(ds['train'].features)

<class 'PIL.JpegImagePlugin.JpegImageFile'>
{'jfif': 257, 'jfif_version': (1, 1), 'jfif_unit': 0, 'jfif_density': (1, 1)}
{'image': Image(mode=None, decode=True), 'text': Value('string')}


In [27]:
# Check if filenames are stored internally
print(ds['train'].cache_files)

[{'filename': '/home/karthi/.cache/huggingface/datasets/Teklia___iam-line/default/0.0.0/fbdad97500ce54635c0d1ba306bf535cb40656cf/iam-line-train.arrow'}]


In [28]:
import pyarrow as pa
import pyarrow.ipc as ipc

path = '/home/karthi/.cache/huggingface/datasets/Teklia___iam-line/default/0.0.0/fbdad97500ce54635c0d1ba306bf535cb40656cf/iam-line-train.arrow'

# Read the arrow file and check all column names
with open(path, 'rb') as f:
    reader = ipc.open_stream(f)
    batch = reader.read_next_batch()
    print("Columns:", batch.schema.names)
    print("First row:", {col: batch.column(col)[0] for col in batch.schema.names})

Columns: ['image', 'text']
First row: {'image': <pyarrow.StructScalar: [('bytes', b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x08\x06\x06\x07\x06\x05\x08\x07\x07\x07\t\t\x08\n\x0c\x14\r\x0c\x0b\x0b\x0c\x19\x12\x13\x0f\x14\x1d\x1a\x1f\x1e\x1d\x1a\x1c\x1c $.\' ",#\x1c\x1c(7),01444\x1f\'9=82<.342\xff\xc0\x00\x0b\x08\x00\x80\t\xa3\x01\x01\x11\x00\xff\xc4\x00\x1f\x00\x00\x01\x05\x01\x01\x01\x01\x01\x01\x00\x00\x00\x00\x00\x00\x00\x00\x01\x02\x03\x04\x05\x06\x07\x08\t\n\x0b\xff\xc4\x00\xb5\x10\x00\x02\x01\x03\x03\x02\x04\x03\x05\x05\x04\x04\x00\x00\x01}\x01\x02\x03\x00\x04\x11\x05\x12!1A\x06\x13Qa\x07"q\x142\x81\x91\xa1\x08#B\xb1\xc1\x15R\xd1\xf0$3br\x82\t\n\x16\x17\x18\x19\x1a%&\'()*456789:CDEFGHIJSTUVWXYZcdefghijstuvwxyz\x83\x84\x85\x86\x87\x88\x89\x8a\x92\x93\x94\x95\x96\x97\x98\x99\x9a\xa2\xa3\xa4\xa5\xa6\xa7\xa8\xa9\xaa\xb2\xb3\xb4\xb5\xb6\xb7\xb8\xb9\xba\xc2\xc3\xc4\xc5\xc6\xc7\xc8\xc9\xca\xd2\xd3\xd4\xd5\xd6\xd7\xd8\xd9\xda\xe1\xe2\xe3\xe4\x

In [29]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from datasets import load_dataset

class IAM_Dataset(Dataset):
    """
    PyTorch Dataset for IAM handwriting lines.

    Each sample returns:
        image     (torch.FloatTensor) : shape (1, 128, 512)
        tokens    (torch.LongTensor)  : shape (max_len,)
        writer_id (int)               : derived from sample index
    """
    def __init__(self, split='train', max_len=256, img_width=512, img_height=128):
        self.data = load_dataset("Teklia/IAM-line")[split]
        self.max_len = max_len
        self.char2idx, self.idx2char = build_vocab()
        self.transform = transforms.Compose([
            transforms.Resize((img_height, img_width)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = sample['image'].convert('L')
        image = self.transform(image)
        tokens = encode_text(sample['text'], self.char2idx, self.max_len)
        writer_id = idx // 20
        return image, tokens, writer_id


# Test
dataset = IAM_Dataset(split='train')
print(f"Dataset size:  {len(dataset)}")

image, tokens, writer_id = dataset[0]
print(f"Image shape:   {image.shape}")
print(f"Tokens shape:  {tokens.shape}")
print(f"Writer ID:     {writer_id}")
print(f"Text:          {dataset.data[0]['text']}")

Dataset size:  6482
Image shape:   torch.Size([1, 128, 512])
Tokens shape:  torch.Size([256])
Writer ID:     0
Text:          put down a resolution on the subject


In [30]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from torchvision import transforms
from datasets import load_dataset

def build_vocab():
    charset = tuple(chr(i) for i in range(32, 126))
    special_tokens = ['<PAD>', '<BOS>', '<EOS>', '<UNK>']
    char2idx = {ch: idx for idx, ch in enumerate(special_tokens)}
    for i, ch in enumerate(charset):
        char2idx[ch] = i + len(special_tokens)
    idx2char = {idx: ch for ch, idx in char2idx.items()}
    return char2idx, idx2char

def encode_text(text, char2idx, max_len=256):
    tokens = [char2idx.get(c, char2idx['<UNK>']) for c in text]
    tokens = tokens[:max_len - 2]
    tokens = [char2idx['<BOS>']] + tokens + [char2idx['<EOS>']]
    pad_len = max_len - len(tokens)
    tokens = tokens + [char2idx['<PAD>']] * pad_len
    return torch.tensor(tokens, dtype=torch.long)

In [31]:
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=4, shuffle=True)

# Grab one batch
batch_images, batch_tokens, batch_writers = next(iter(loader))

print(f"Batch image shape:  {batch_images.shape}")   # (4, 1, 128, 512)
print(f"Batch tokens shape: {batch_tokens.shape}")   # (4, 256)
print(f"Batch writers:      {batch_writers}")        # 4 writer IDs

Batch image shape:  torch.Size([4, 1, 128, 512])
Batch tokens shape: torch.Size([4, 256])
Batch writers:      tensor([192, 251, 180, 237])


In [32]:
code = '''import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from models.text_encoder import build_vocab, encode_text


class IAM_Dataset(Dataset):
    """
    PyTorch Dataset for IAM handwriting lines.

    Each sample returns:
        image     (torch.FloatTensor) : shape (1, 128, 512)
        tokens    (torch.LongTensor)  : shape (max_len,)
        writer_id (int)               : derived from sample index
    """
    def __init__(self, split="train", max_len=256, img_width=512, img_height=128):
        self.data = load_dataset("Teklia/IAM-line")[split]
        self.max_len = max_len
        self.char2idx, self.idx2char = build_vocab()
        self.transform = transforms.Compose([
            transforms.Resize((img_height, img_width)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        image = sample["image"].convert("L")
        image = self.transform(image)
        tokens = encode_text(sample["text"], self.char2idx, self.max_len)
        writer_id = idx // 20
        return image, tokens, writer_id


def get_dataloader(split="train", batch_size=16, shuffle=True):
    """
    Returns a DataLoader for the IAM dataset.

    Args:
        split      (str)  : train, validation, or test
        batch_size (int)  : number of samples per batch
        shuffle    (bool) : shuffle the dataset
    Returns:
        DataLoader
    """
    dataset = IAM_Dataset(split=split)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
'''

with open("../models/dataset.py", "w") as f:
    f.write(code)

print("Saved to models/dataset.py ✅")

Saved to models/dataset.py ✅


In [33]:
try:
    import diffusers
    print(f"diffusers: {diffusers.__version__} ✅")
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "diffusers", "accelerate", "transformers"], check=True)
    print("Done ✅")

  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 1.9 MB/s  0:00:02m 1.8 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 3.6 MB/s  0:00:023.6 MB/s eta 0:00:0101
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.5/801.5 kB 3.5 MB/s  0:00:00? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [transformers]0m 7/8 [transformers]
Done ✅
